# 模块 3 第 2 节：通过 SDK 与部署交互

我们的 Agent 现已部署到生产中并进行适当的监控。但我们如何真正**使用**它呢？

在本节中，我们将学习如何使用 **LangGraph SDK** 以编程方式与部署的 Agent 进行交互。这使您能够：
- 构建自定义用户界面和应用程序
- 自动化测试和工作流程
- 处理human-in-the-loop中断
- 与现有系统集成

最后，您将能够从任何 Python 应用程序调用已部署的 Agent。

## 设置

In [ ]:
from dotenv import load_dotenv
from langgraph_sdk import get_client
from langgraph_sdk.schema import Command

load_dotenv()

# Connect to your deployment
DEPLOYMENT_API_URL = "your-deployment-api-url"
client = get_client(url=DEPLOYMENT_API_URL)

---
## 1. 基本调用

让我们从简单开始 - 调用我们部署的 Agent 并获得响应。

我们将询问产品信​​息（无需身份验证）。


**注意：** 第一次调用使用向量搜索的子Agent将按需构建向量数据库，一次性设置。

In [2]:
# Create a thread (conversation session)
thread = await client.threads.create()

# Send a message
input_message = {
    "messages": [
        {
            "role": "user",
            "content": "What are the key features of the Sony WH-1000XM5 headphones?",
        }
    ]
}

# Invoke the agent (wait for completion)
response = await client.runs.wait(
    thread["thread_id"],
    "supervisor_hitl_sql_agent",  # Your deployed graph name
    input=input_message,
)

# Print the final response
final_message = response["messages"][-1]["content"]
print(f"Agent: {final_message}")

Agent: Based on our product documentation, here are the **key features of the Sony WH-1000XM5 headphones**:

**Noise Cancellation:**
- Industry-leading Active Noise Cancellation (ANC) with 8 microphones
- HD Noise Cancelling Processor QN1 that adjusts 700 times per second
- Excellent at blocking low-frequency noise (planes, traffic) and mid-frequency sounds (conversations, office noise)
- NC Optimizer feature personalizes cancellation to your ear shape and fit

**Audio Quality:**
- Premium 30mm drivers delivering rich, detailed sound
- Compatible with all Bluetooth devices (iOS, Android, Windows, Mac)

**Battery & Connectivity:**
- **30-hour battery life** (exceptional for wireless headphones)
- Seamless multi-device connectivity

**Microphone & Call Quality:**
- 4 beamforming microphones for clear call quality
- 4 ANC microphones

**Smart Controls:**
- Touch-sensitive controls on the right earcup: tap to play/pause, double-tap to skip, swipe for volume
- Quick Attention mode to tempor

**刚刚发生了什么？**
1. 创建了一个新的thread（就像开始一个新的对话）
2. 发送消息至Agent
3. 等待Agent完成
4. 检索最终消息

**注意：** 检查 LangSmith - 您将在 `langsmith-agent-lifecycle-workshop-deployment` 项目中看到此 trace！

---
## 2. Human-in-the-Loop (HITL) 与 SDK

我们的 Agent 需要电子邮件验证来解决个人帐户问题。让我们看看如何以编程方式处理**中断**。

### 第 1 步：开始需要验证的对话

**注意：** LangSmith Studio 自动在其 UI 中处理 interrupt 恢复，但以编程方式使用 SDK 时，您需要显式使用 `Command(resume=...)` 模式（请参阅步骤 3）。

In [10]:
# Create a new thread
thread = await client.threads.create()
thread_id = thread["thread_id"]

# Ask about order status (requires email verification)
input_message = {
    "messages": [{"role": "user", "content": "What's the status of my recent order?"}]
}

# Run the graph until the interrupt is hit
result = await client.runs.wait(
    thread_id,
    "supervisor_hitl_sql_agent",
    input=input_message,
)

# Print the interrupt
print(result["__interrupt__"])

[{'id': '23754cfd4d40de50b188c85d3bbd4cee', 'value': 'Please provide your email:'}]


### 步骤 2：检查 Interrupt 并获取状态

In [11]:
# Get the current state
state = await client.threads.get_state(thread_id)

# The agent is waiting for email input
# Check the last message
last_message = state["values"]["messages"][-1]["content"]
print(f"Agent says: {last_message}")

Agent says: To access information about your account or orders, please provide your email address.


### 第 3 步：提供邮箱并恢复执行

**重要：** 要恢复中断的运行：
1. 在**同一个thread**上再次拨打`client.runs.wait()`
2. 将 `command=Command(resume=...)` 与用户的输入（作为字符串）一起传递
3. Agent 从 interrupt 点以提供的值恢复

**关键：** `Command(resume=...)` 告诉 SDK 恢复中断的运行而不是开始新的运行。您传递给 `resume` 的值将成为图形代码中 `interrupt()` 调用的返回值。

In [14]:
# Resume the graph by passing the email as a Command
# Note: We pass the user's message content directly to Command(resume=...)
response = await client.runs.wait(
    thread_id,
    "supervisor_hitl_sql_agent",
    command=Command(resume="My email is consultant.tech@gmail.com"),
)

# Get the final answer
final_message = response["messages"][-1]["content"]
print(f"\nAgent: {final_message}")


Agent: Your recent order **ORD-2025-0044** is currently in **Processing** status. Here are the details:

- **Order Date:** October 15, 2025
- **Total Amount:** $549.00
- **Status:** Processing
- **Shipped Date:** Not yet shipped
- **Tracking Number:** Not yet available

Your order is being prepared and will be shipped soon. Once it's dispatched, you'll receive tracking information so you can monitor its delivery progress. Is there anything else you'd like to know about this order?


### 第 4 步：询问后续跟进（这次没有 Interrupt！）

In [15]:
# Ask a follow-up question (customer_id already verified!)
followup_message = {
    "messages": [{"role": "user", "content": "What exactly was in that order?"}]
}

response = await client.runs.wait(
    thread_id,
    "supervisor_hitl_sql_agent",
    input=followup_message,
)

final_message = response["messages"][-1]["content"]
print(f"Agent: {final_message}")

Agent: Your order **ORD-2025-0044** contains:

- **1x Dell UltraSharp 27" 4K Monitor** - $549.00

This is a high-quality 4K monitor, and it's currently in stock. Your order is being prepared for shipment. Is there anything else you'd like to know about your order or this product?


**发生了什么？**
1. 第一个问题触发邮件验证（interrupt）
2. 我们恢复`Command(resume=...)`提供电子邮件
3. 后续问题没有interrupt - `customer_id`已经处于thread状态！

**关键 HITL 模式：**
- 在同一个 thread 上使用 `command=Command(resume=value)` 来恢复 interrupt
- 该值成为图表中 `interrupt()` 的返回值
- Thread 状态（如 `customer_id`）在同一 thread 上的运行中保持不变

这就是您使用 HITL 流构建自定义 UI 的方法。

**了解更多：** [HITL 和 SDK 文档](https://docs.langchain.com/langsmith/add-human-in-the-loop)

---
## 3. 监控生产trace

每个 SDK 调用都会在 LangSmith 中创建一个 trace。让我们验证一下我们的在线 evaluator 是否正常工作。

### 检查 LangSmith 的trace

**步骤：**
1. 转到 **LangSmith** → **项目** → **langsmith-Agent-lifecycle-workshop-deployment**
2. 您将看到 SDK 调用的所有trace
3、点击任意trace即可看到：
   - 完整的对话历史记录
   - Tool Calling和推理
   - 延迟指标
   - 任何错误或警告

**1 分钟后**（您的 thread 空闲时间），检查每个 trace 上的 **反馈** 选项卡：
- 您应该看到 `user_sentiment: positive`（因为我们进行了有用的对话）
- 在线evaluator自动运行！